# AIC 2026 — Visual embedding ingestion · BEiT-3 Large COCO Retrieval

**Input:** `aqpahm/aic2026-keyframes-transnetv2`  
**Output:** `aqpahm/aic2026-visual-beit3-large-coco-retrieval`  
**Model:** Microsoft BEiT-3 Large Patch16 384, fine-tuned for COCO image–text retrieval

Notebook nguồn chạy độc lập trên **Google Colab hoặc Kaggle**. Notebook tải mã nguồn
và checkpoint chính thức đã ghim phiên bản, dùng toàn bộ GPU hiện có, tự giảm batch
khi thiếu VRAM, upload theo nhóm và tiếp tục an toàn sau khi gián đoạn.

Embedding được ingest độc lập. Notebook này chưa quyết định cách index, fusion hoặc
kiến trúc retrieval. Khi truy vấn sau này, text embedding phải được tạo bằng đúng
checkpoint và tokenizer BEiT-3 tương ứng.


In [ ]:
%pip install -q -U "huggingface_hub>=0.34,<2" "safetensors>=0.4" "pyarrow>=16" "requests>=2.31" "timm==0.9.16" "torchscale==0.2.0"


In [ ]:
import os
import sys
import tempfile

# Set these before importing huggingface_hub.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"

import gc
import hashlib
import io
import json
import re
import shutil
import tarfile
import time
import types
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from queue import Queue

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download
from PIL import Image, ImageFile
from safetensors import safe_open
from safetensors.torch import save_file
from torchvision import transforms
from torchvision.transforms import InterpolationMode

ImageFile.LOAD_TRUNCATED_IMAGES = True

INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
OUTPUT_REPO = "aqpahm/aic2026-visual-beit3-large-coco-retrieval"
MODEL_ID = "microsoft/beit3-large-patch16-384-coco-retrieval"
MODEL_SOURCE_REPO = "microsoft/unilm"
MODEL_SOURCE_REVISION = "ca43e4cd19445a536f133bf2bc25b573b2f0c7c5"
MODEL_SOURCE_SHA256 = {
    "modeling_utils.py": "cc04b762b2c0ba32eb82d65a5e543adada156ce3584a351b56c87eb3de293a27",
    "modeling_finetune.py": "dfa9a2dbba7e7b2a46023da0d693f6e781d946d86b2b7a52e69ed4bd463d763f",
}
CHECKPOINT_URL = (
    "https://github.com/addf400/files/releases/download/beit3/"
    "beit3_large_patch16_384_coco_retrieval.pth"
)
CHECKPOINT_FILENAME = "beit3_large_patch16_384_coco_retrieval.pth"
CHECKPOINT_SIZE_BYTES = 1_350_590_595
TOKENIZER_URL = "https://github.com/addf400/files/releases/download/beit3/beit3.spm"

# BEiT-3 Large is heavier than SigLIP. A T4 starts conservatively and OOM recovery
# halves the batch automatically. Increase only after observing a representative pilot.
INITIAL_BATCH_SIZE = 16
UPLOAD_BATCH_VIDEOS = 8
MAX_VIDEOS_PER_RUN = None  # set an integer only for a deliberate pilot
OUTPUT_PRIVATE = True
STRICT_REMOTE_VALIDATION = False  # True downloads and validates every marker
EXPECTED_DIMENSION = 1024
IMAGE_SIZE = 384
SCHEMA_VERSION = 1


def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata

        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient

        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(
            f"Missing {name}. Add it as a Colab/Kaggle secret and enable notebook access."
        )
    return value


def hosted_work_root(job_name):
    if RUNTIME == "colab":
        return Path("/content") / job_name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / job_name
    return Path(tempfile.gettempdir()) / job_name


RUNTIME = detect_runtime()
ROOT = hosted_work_root("aic-visual-beit3-large")
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
MODEL_ROOT = ROOT / "model"
SOURCE_ROOT = MODEL_ROOT / "unilm-beit3"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT, MODEL_ROOT, SOURCE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

HF_TOKEN = read_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME)
print("Hugging Face:", account["name"])

api.create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=OUTPUT_PRIVATE,
    exist_ok=True,
)
INPUT_REVISION = api.dataset_info(INPUT_REPO, token=HF_TOKEN).sha
print("Input revision:", INPUT_REVISION)
print("Model source revision:", MODEL_SOURCE_REVISION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in the Colab/Kaggle runtime before running this job")

GPU_IDS = list(range(torch.cuda.device_count()))
DEVICES = [torch.device(f"cuda:{device_id}") for device_id in GPU_IDS]
BATCH_SIZE_BY_GPU = {}
for device_id in GPU_IDS:
    properties = torch.cuda.get_device_properties(device_id)
    vram_gib = properties.total_memory / 1024**3
    initial = INITIAL_BATCH_SIZE if vram_gib >= 13 else max(2, INITIAL_BATCH_SIZE // 2)
    BATCH_SIZE_BY_GPU[device_id] = initial
    print(f"cuda:{device_id}: {properties.name}, {vram_gib:.1f} GiB, initial batch {initial}")
print("GPU workers:", len(GPU_IDS))
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
torch.backends.cudnn.benchmark = True


In [ ]:
def retry(operation, description, attempts=7):
    """Retry network operations with capped exponential backoff."""
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(300, 5 * (2 ** (attempt - 1)))
            print(
                f"{description} failed ({type(error).__name__}); "
                f"retry in {delay}s [{attempt}/{attempts}]"
            )
            time.sleep(delay)


def download_http_file(url, destination, expected_size=None, attempts=7):
    """Resumable HTTP download with an atomic final rename."""
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + ".part")
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and (expected_size is None or destination.stat().st_size == expected_size):
        return destination
    if destination.exists():
        destination.unlink()

    for attempt in range(1, attempts + 1):
        try:
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            with requests.get(url, headers=headers, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                append = offset > 0 and response.status_code == 206
                if offset > 0 and not append:
                    offset = 0
                mode = "ab" if append else "wb"
                with partial.open(mode) as file:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            file.write(chunk)
            size = partial.stat().st_size
            if expected_size is not None and size != expected_size:
                raise RuntimeError(f"downloaded {size} bytes, expected {expected_size}")
            partial.replace(destination)
            return destination
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(120, 5 * (2 ** (attempt - 1)))
            print(f"checkpoint download failed: {error!r}; retry in {delay}s")
            time.sleep(delay)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as file:
        while chunk := file.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()


def download_source_file(url, destination):
    def operation():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        payload = response.content
        if len(payload) < 1_000:
            raise RuntimeError(f"Invalid source payload from {url}")
        temporary = destination.with_suffix(destination.suffix + ".part")
        temporary.write_bytes(payload)
        temporary.replace(destination)
        return destination

    expected_hash = MODEL_SOURCE_SHA256[destination.name]
    if not destination.exists() or sha256_file(destination) != expected_hash:
        destination.unlink(missing_ok=True)
        retry(operation, f"download {destination.name}")
    actual_hash = sha256_file(destination)
    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Source checksum mismatch for {destination.name}: {actual_hash}"
        )
    return destination


# Fetch only the two inference source files from Microsoft's pinned UniLM commit.
for filename in ("modeling_utils.py", "modeling_finetune.py"):
    url = (
        f"https://raw.githubusercontent.com/{MODEL_SOURCE_REPO}/"
        f"{MODEL_SOURCE_REVISION}/beit3/{filename}"
    )
    download_source_file(url, SOURCE_ROOT / filename)

checkpoint_path = download_http_file(
    CHECKPOINT_URL,
    MODEL_ROOT / CHECKPOINT_FILENAME,
    expected_size=CHECKPOINT_SIZE_BYTES,
)
CHECKPOINT_SHA256 = sha256_file(checkpoint_path)
MODEL_REVISION = CHECKPOINT_SHA256
print("Checkpoint SHA-256:", CHECKPOINT_SHA256)

# modeling_finetune imports `utils`, but inference only needs these three symbols.
# A small shim avoids importing BEiT-3's obsolete training stack on modern runtimes.
class InferenceOnlyClipLoss(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

    def forward(self, *args, **kwargs):
        raise RuntimeError("Training loss is unavailable in this inference-only notebook")


utils_shim = types.ModuleType("utils")
utils_shim.ClipLoss = InferenceOnlyClipLoss
utils_shim.get_rank = lambda: 0
utils_shim.get_world_size = lambda: 1
sys.modules["utils"] = utils_shim
sys.path.insert(0, str(SOURCE_ROOT))

from modeling_finetune import beit3_large_patch16_384_retrieval

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
checkpoint_state = checkpoint.get("model", checkpoint)
models = []
for device in DEVICES:
    current_model = beit3_large_patch16_384_retrieval(pretrained=False)
    current_model.load_state_dict(checkpoint_state, strict=True)
    current_model = current_model.to(device=device, dtype=torch.float16).eval()
    for parameter in current_model.parameters():
        parameter.requires_grad_(False)
    models.append(current_model)
del checkpoint, checkpoint_state
gc.collect()

image_transform = transforms.Compose(
    [
        transforms.Resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=InterpolationMode.BICUBIC,
            antialias=True,
        ),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)

# Fail early if the current cloud runtime is incompatible with the pinned code/checkpoint.
for model_index, (active_model, device) in enumerate(zip(models, DEVICES)):
    probe = torch.zeros((1, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.float16, device=device)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        vision_features, _ = active_model(image=probe, only_infer=True)
    if tuple(vision_features.shape) != (1, EXPECTED_DIMENSION):
        raise RuntimeError(f"BEiT-3 smoke test failed: {tuple(vision_features.shape)}")
    del probe, vision_features
    torch.cuda.empty_cache()
    print(f"Loaded and validated BEiT-3 replica {model_index} on {device}")


In [ ]:
def download_hf_file(filename, local_dir):
    """Download a pinned input file and reuse partial/cache data on retry."""
    local_dir.mkdir(parents=True, exist_ok=True)
    return Path(
        retry(
            lambda: hf_hub_download(
                repo_id=INPUT_REPO,
                repo_type="dataset",
                filename=filename,
                revision=INPUT_REVISION,
                token=HF_TOKEN,
                local_dir=local_dir,
            ),
            f"download {filename}",
        )
    )


def normalized_tar_name(value):
    return PurePosixPath(str(value).replace("\\", "/")).as_posix().lstrip("./")


def build_tar_lookup(archive):
    by_name = {}
    by_basename = defaultdict(list)
    for member in archive.getmembers():
        if not member.isfile():
            continue
        name = normalized_tar_name(member.name)
        by_name[name] = member
        by_basename[PurePosixPath(name).name].append(member)
    return by_name, by_basename


def locate_tar_member(image_path, by_name, by_basename):
    expected = normalized_tar_name(image_path)
    if expected in by_name:
        return by_name[expected]
    basename = PurePosixPath(expected).name
    matches = by_basename.get(basename, [])
    if len(matches) != 1:
        raise RuntimeError(f"Cannot uniquely locate {image_path!r} inside keyframes.tar")
    return matches[0]


def read_image_batch(archive, records, by_name, by_basename):
    tensors = []
    for record in records:
        member = locate_tar_member(record["image_path"], by_name, by_basename)
        stream = archive.extractfile(member)
        if stream is None:
            raise RuntimeError(f"Cannot read tar member {member.name}")
        with Image.open(io.BytesIO(stream.read())) as source:
            image = source.convert("RGB")
            image.load()
        tensors.append(image_transform(image))
        image.close()
    return torch.stack(tensors, dim=0)


def image_features(pixel_values, active_model, device):
    pixel_values = pixel_values.to(device=device, dtype=torch.float16, non_blocking=True)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        features, _ = active_model(image=pixel_values, only_infer=True)
    if not torch.is_tensor(features):
        raise RuntimeError(f"Unexpected feature type: {type(features).__name__}")
    features = torch.nn.functional.normalize(features.float(), dim=-1)
    return features.half().cpu()


def embed_archive(archive, records, initial_batch_size, active_model, device):
    """Embed aligned tar members; reduce batch size after CUDA OOM."""
    by_name, by_basename = build_tar_lookup(archive)
    chunks = []
    cursor = 0
    active_batch = min(initial_batch_size, len(records))
    minimum_batch = active_batch

    while cursor < len(records):
        selected = records[cursor : cursor + active_batch]
        pixel_values = read_image_batch(archive, selected, by_name, by_basename)
        try:
            chunk = image_features(pixel_values, active_model, device)
        except (torch.OutOfMemoryError, RuntimeError) as error:
            is_oom = isinstance(error, torch.OutOfMemoryError) or "out of memory" in str(error).lower()
            if not is_oom or active_batch == 1:
                raise
            active_batch = max(1, active_batch // 2)
            minimum_batch = min(minimum_batch, active_batch)
            print(f"CUDA OOM; reducing batch size to {active_batch}")
            gc.collect()
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
            continue
        finally:
            del pixel_values

        if chunk.ndim != 2 or chunk.shape[1] != EXPECTED_DIMENSION:
            raise RuntimeError(
                f"Expected [N, {EXPECTED_DIMENSION}] embeddings, got {tuple(chunk.shape)}"
            )
        chunks.append(chunk)
        cursor += len(selected)
        if cursor % 256 < len(selected) or cursor == len(records):
            print(f"  embedded {cursor}/{len(records)}")

    embeddings = torch.cat(chunks, dim=0)
    if len(embeddings) != len(records):
        raise RuntimeError(f"Embedding count mismatch: {len(embeddings)} vs {len(records)}")
    if not torch.isfinite(embeddings).all():
        raise RuntimeError("Embedding tensor contains NaN or infinity")
    norms = torch.linalg.vector_norm(embeddings.float(), dim=1)
    max_norm_error = float((norms - 1).abs().max())
    if max_norm_error > 0.005:
        raise RuntimeError(f"L2 normalization check failed: max error {max_norm_error}")
    return embeddings.contiguous(), minimum_batch, max_norm_error


In [ ]:
IDENTITY_COLUMNS = [
    "video_id",
    "frame_uid",
    "sample_n",
    "frame_idx",
    "shot_id",
    "timestamp_sec",
    "image_path",
    "embedding_row",
]


def make_manifest(metadata, video_id):
    required = {"video_id", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"}
    missing = required - set(metadata.columns)
    if missing:
        raise RuntimeError(f"{video_id}: source columns missing: {sorted(missing)}")
    if metadata.empty:
        raise RuntimeError(f"{video_id}: source metadata is empty")

    manifest = metadata.sort_values("sample_n", kind="stable").reset_index(drop=True).copy()
    if manifest["video_id"].astype(str).nunique() != 1:
        raise RuntimeError(f"{video_id}: metadata contains multiple video IDs")
    if str(manifest.iloc[0]["video_id"]) != video_id:
        raise RuntimeError(f"{video_id}: source video ID does not match path")
    if manifest["sample_n"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate sample_n values")
    if manifest["frame_idx"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate frame_idx values")

    manifest["video_id"] = manifest["video_id"].astype(str)
    manifest["sample_n"] = manifest["sample_n"].astype("int64")
    manifest["frame_idx"] = manifest["frame_idx"].astype("int64")
    manifest["shot_id"] = manifest["shot_id"].astype("int64")
    manifest["timestamp_sec"] = manifest["timestamp_sec"].astype("float64")
    manifest["image_path"] = manifest["image_path"].astype(str)
    manifest["frame_uid"] = [
        f"{video_id}:{frame_idx}"
        for frame_idx in manifest["frame_idx"].tolist()
    ]
    manifest["embedding_row"] = np.arange(len(manifest), dtype=np.int64)
    return manifest[IDENTITY_COLUMNS]


def catalog_hash(manifest):
    hasher = hashlib.sha256()
    for row in manifest.itertuples(index=False):
        payload = {
            "video_id": row.video_id,
            "frame_uid": row.frame_uid,
            "sample_n": int(row.sample_n),
            "frame_idx": int(row.frame_idx),
            "shot_id": int(row.shot_id),
            "timestamp_sec": float(row.timestamp_sec),
            "image_path": row.image_path,
            "embedding_row": int(row.embedding_row),
        }
        hasher.update(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8"))
        hasher.update(b"\n")
    return hasher.hexdigest()



def write_video_output(spec, manifest, embeddings, stats, max_norm_error):
    directory = OUTPUT_ROOT / spec["level"] / spec["video_id"]
    shutil.rmtree(directory, ignore_errors=True)
    directory.mkdir(parents=True, exist_ok=True)

    embedding_path = directory / "embeddings.safetensors"
    frames_path = directory / "frames.parquet"
    marker_path = directory / "_VISUAL_SUCCESS.json"

    save_file(
        {"embeddings": embeddings},
        str(embedding_path),
        metadata={
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "normalized": "l2",
            "dtype": "float16",
            "schema_version": str(SCHEMA_VERSION),
        },
    )
    manifest.to_parquet(frames_path, index=False)

    # Read the stored tensor back before publishing the success marker.
    with safe_open(str(embedding_path), framework="pt", device="cpu") as file:
        stored = file.get_tensor("embeddings")
    if tuple(stored.shape) != (len(manifest), EXPECTED_DIMENSION):
        raise RuntimeError(f"Stored tensor has invalid shape: {tuple(stored.shape)}")
    if stored.dtype != torch.float16:
        raise RuntimeError(f"Stored tensor has invalid dtype: {stored.dtype}")

    marker = {
        "video_id": spec["video_id"],
        "status": "success",
        "frames": len(manifest),
        "embedding_dimension": EXPECTED_DIMENSION,
        "embedding_dtype": "float16",
        "normalization": "l2",
        "max_norm_error_after_fp16": max_norm_error,
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "source_tar": spec["tar_filename"],
        "source_frames": spec["frames_filename"],
        "catalog_sha256": catalog_hash(manifest),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "model_source_repo": MODEL_SOURCE_REPO,
        "model_source_revision": MODEL_SOURCE_REVISION,
        "checkpoint_url": CHECKPOINT_URL,
        "checkpoint_sha256": CHECKPOINT_SHA256,
        "tokenizer_url": TOKENIZER_URL,
        "preprocessing": "resize_384_bicubic_rgb_normalize_0.5",
        "image_size": IMAGE_SIZE,
        "embeddings_sha256": sha256_file(embedding_path),
        "stats": stats,
        "schema_version": SCHEMA_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [embedding_path, frames_path, marker_path]


def upload_outputs(paths, video_ids):
    operations = [
        CommitOperationAdd(
            path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
            path_or_fileobj=str(path),
        )
        for path in paths
    ]
    retry(
        lambda: api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add BEiT-3 Large embeddings for {video_ids[0]} through {video_ids[-1]}",
            token=HF_TOKEN,
        ),
        f"upload {len(video_ids)} videos",
    )


In [ ]:
input_files = set(
    retry(
        lambda: api.list_repo_files(INPUT_REPO, repo_type="dataset", revision=INPUT_REVISION),
        "list input files",
    )
)
output_files = set(
    retry(
        lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
        "list output files",
    )
)

tar_pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = tar_pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    frames_filename = f"data/{level}/{video_id}/frames.parquet"
    if frames_filename not in input_files:
        raise RuntimeError(f"Missing source metadata: {frames_filename}")
    video_specs.append(
        {
            "level": level,
            "video_id": video_id,
            "tar_filename": filename,
            "frames_filename": frames_filename,
        }
    )


def remote_complete(spec, files):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    marker_remote = f"{prefix}/_VISUAL_SUCCESS.json"
    required = ("embeddings.safetensors", "frames.parquet", "_VISUAL_SUCCESS.json")
    if not all(f"{prefix}/{name}" in files for name in required):
        return False
    if not STRICT_REMOTE_VALIDATION:
        return True
    try:
        marker_path = hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            filename=marker_remote,
            token=HF_TOKEN,
            force_download=True,
        )
        payload = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        return (
            payload.get("status") == "success"
            and payload.get("model_id") == MODEL_ID
            and payload.get("model_revision") == MODEL_REVISION
            and payload.get("source_revision") == INPUT_REVISION
            and int(payload.get("embedding_dimension", 0)) == EXPECTED_DIMENSION
            and int(payload.get("frames", 0)) > 0
            and int(payload.get("schema_version", 0)) == SCHEMA_VERSION
        )
    except Exception as error:
        print(f"Remote marker invalid for {spec['video_id']}: {error!r}")
        return False


completed_specs = [spec for spec in video_specs if remote_complete(spec, output_files)]
pending_specs = [spec for spec in video_specs if not remote_complete(spec, output_files)]
if MAX_VIDEOS_PER_RUN is not None:
    pending_specs = pending_specs[:MAX_VIDEOS_PER_RUN]

print("Source videos:", len(video_specs))
print("Already complete:", len(completed_specs))
print("Selected this run:", len(pending_specs))
if not video_specs:
    raise RuntimeError(
        f"No keyframe TAR files found in {INPUT_REPO}. Check INPUT_REPO after renaming."
    )


In [ ]:
def process_video(spec):
    model_index = device_pool.get()
    device_id = GPU_IDS[model_index]
    device = DEVICES[model_index]
    video_id = spec["video_id"]
    task_dir = DOWNLOAD_ROOT / video_id
    shutil.rmtree(task_dir, ignore_errors=True)
    task_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device_id)
    try:
        frames_path = download_hf_file(spec["frames_filename"], task_dir)
        tar_path = download_hf_file(spec["tar_filename"], task_dir)
        manifest = make_manifest(pd.read_parquet(frames_path), video_id)
        records = manifest.to_dict("records")

        with tarfile.open(tar_path, mode="r:*") as archive:
            embeddings, minimum_batch, max_norm_error = embed_archive(
                archive,
                records,
                BATCH_SIZE_BY_GPU[device_id],
                models[model_index],
                device,
            )

        torch.cuda.synchronize(device_id)
        elapsed = time.perf_counter() - started
        stats = {
            "elapsed_sec": round(elapsed, 3),
            "frames_per_sec": round(len(manifest) / max(elapsed, 1e-6), 3),
            "peak_vram_gib": round(torch.cuda.max_memory_allocated(device_id) / 1024**3, 3),
            "minimum_batch_size": minimum_batch,
            "device": f"cuda:{device_id}",
            "gpu": torch.cuda.get_device_name(device_id),
        }
        paths = write_video_output(spec, manifest, embeddings, stats, max_norm_error)
        return {"video_id": video_id, "paths": paths, "frames": len(manifest), "stats": stats}
    finally:
        shutil.rmtree(task_dir, ignore_errors=True)
        gc.collect()
        with torch.cuda.device(device_id):
            torch.cuda.empty_cache()
        device_pool.put(model_index)


pending_paths = []
pending_ids = []
run_results = []
failures = []
device_pool = Queue()
for model_index in range(len(models)):
    device_pool.put(model_index)


def flush_pending():
    global pending_paths, pending_ids
    if not pending_paths:
        return
    upload_outputs(pending_paths, pending_ids)
    print(f"Uploaded {len(pending_ids)} videos in one commit")
    for video_id in pending_ids:
        level = video_id.split("_")[0]
        shutil.rmtree(OUTPUT_ROOT / level / video_id, ignore_errors=True)
    pending_paths = []
    pending_ids = []


run_started = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(models)) as executor:
    futures = {executor.submit(process_video, spec): spec for spec in pending_specs}
    for position, future in enumerate(as_completed(futures), 1):
        spec = futures[future]
        try:
            result = future.result()
            run_results.append(result)
            pending_paths.extend(result["paths"])
            pending_ids.append(result["video_id"])
            stats = result["stats"]
            print(
                f"[{position}/{len(futures)}] OK {result['video_id']}: "
                f"{result['frames']} frames, {stats['frames_per_sec']} FPS, "
                f"{stats['peak_vram_gib']} GiB, {stats['device']}"
            )
            if len(pending_ids) >= UPLOAD_BATCH_VIDEOS:
                flush_pending()
        except Exception as error:
            failures.append({"video_id": spec["video_id"], "error": repr(error)})
            print(f"[{position}/{len(futures)}] FAILED {spec['video_id']}: {error!r}")

flush_pending()
print("=" * 72)
print("GPU workers:", len(models))
print("Processed this run:", len(run_results))
print("Failures:", len(failures))
if run_results:
    total_frames = sum(result["frames"] for result in run_results)
    wall_seconds = time.perf_counter() - run_started
    print("Frames:", total_frames)
    print("Approximate parallel FPS:", round(total_frames / max(wall_seconds, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError(
        "BEiT-3 Large run has failures. Rerun; valid remote completed videos are skipped."
    )


In [ ]:
# Final remote audit. A pilot intentionally reports the remaining videos.
final_files = set(
    retry(
        lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
        "refresh output files",
    )
)
missing_videos = [
    spec["video_id"] for spec in video_specs if not remote_complete(spec, final_files)
]

print(f"Remote complete: {len(video_specs) - len(missing_videos)}/{len(video_specs)} videos")
print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
if missing_videos:
    print("Remaining:", len(missing_videos), "sample:", missing_videos[:20])
if MAX_VIDEOS_PER_RUN is None and missing_videos:
    raise RuntimeError(f"Missing visual outputs for {len(missing_videos)} videos")

if MAX_VIDEOS_PER_RUN is not None:
    print(
        "Pilot finished. Inspect the output, then set MAX_VIDEOS_PER_RUN = None "
        "and rerun all cells. Completed videos will be skipped."
    )
